In [ ]:
import cv2
import numpy as np
import pygame
import time
from djitellopy import tello
import math

pygame.init()
win = pygame.display.set_mode((400, 400))

drone = tello.Tello()
drone.connect()

try:
    print(f"Battery = {drone.get_battery()}")
except BaseException:
    print(f"Battery = 100 %")  

MAP_SIZE = 600
GRID_INTERVAL = 50

start_x, start_y = MAP_SIZE // 2, MAP_SIZE // 2
x_pos, y_pos = float(start_x), float(start_y)

current_angle = 0.0

SPEED_CM_PER_SEC = 40      
ROTATION_DEG_PER_SEC = 45  

flight_path = [(int(x_pos), int(y_pos))]
last_time = time.time()
drone.takeoff()

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            drone.land()
            pygame.quit()
            exit()

    current_time = time.time()
    dt = current_time - last_time
    last_time = current_time

    keys = pygame.key.get_pressed()
    
    lr, fb, ud, yaw = 0, 0, 0, 0
    moving, rotating, move_direction, rot_direction = False, False, 1, 1

    if keys[pygame.K_w]:
        fb, moving, move_direction = 50, True, 1
    elif keys[pygame.K_s]:
        fb, moving, move_direction = -50, True, -1

    if keys[pygame.K_a]: lr = -50
    elif keys[pygame.K_d]: lr = 50

    if keys[pygame.K_UP]: ud = 50
    elif keys[pygame.K_DOWN]: ud = -50

    if keys[pygame.K_LEFT]:
        yaw , rotating, rot_direction = -50, True, -1
        rotating = True
        rot_direction = -1  
    elif keys[pygame.K_RIGHT]:
        yaw , rotating, rot_direction = 50, True, 1

    drone.send_rc_control(lr, ud, fb, yaw)

    if rotating:
        current_angle += rot_direction * ROTATION_DEG_PER_SEC * dt
        current_angle %= 360

    if moving:
        rad = math.radians(current_angle - 90)
        distance_moved = move_direction * SPEED_CM_PER_SEC * dt
        
        x_pos += distance_moved * math.cos(rad)
        y_pos += distance_moved * math.sin(rad)
        flight_path.append((int(x_pos), int(y_pos)))

    grid_canvas = np.ones((MAP_SIZE, MAP_SIZE, 3), np.uint8) * 240

    for i in range(0, MAP_SIZE, GRID_INTERVAL):
        cv2.line(grid_canvas, (i, 0), (i, MAP_SIZE), (200, 220, 240), 2)
        cv2.line(grid_canvas, (0, i), (MAP_SIZE, i), (200, 220, 240), 2)

    cv2.circle(grid_canvas, (start_x, start_y), 5, (100, 100, 100), cv2.FILLED)

    for p in range(1, len(flight_path)):
        cv2.line(grid_canvas, flight_path[p-1], flight_path[p], (255, 100, 50), 3)

    curr_x, curr_y = int(x_pos), int(y_pos)
    cv2.circle(grid_canvas, (curr_x, curr_y), 10, (50, 130, 240), cv2.FILLED)

    arrow_rad = math.radians(current_angle - 90)
    endpoint_x = int(curr_x + 20 * math.cos(arrow_rad))
    endpoint_y = int(curr_y + 20 * math.sin(arrow_rad))
    cv2.line(grid_canvas, (curr_x, curr_y), (endpoint_x, endpoint_y), (0, 0, 255), 3)

    calc_dist_x = int(x_pos - start_x)
    calc_dist_y = int(start_y - y_pos)
    cv2.putText(grid_canvas, f"Distance X : {calc_dist_x} cm", (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (50, 50, 50), 2)
    cv2.putText(grid_canvas, f"Distance Y : {calc_dist_y} cm", (20, 70), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (50, 50, 50), 2)
    cv2.putText(grid_canvas, f"Angle : {int(current_angle)} deg", (20, 100), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (50, 50, 50), 2)

    cv2.imshow("Drone Flight Mapping System", grid_canvas)
    cv2.waitKey(1)

    if keys[pygame.K_ESCAPE]:
        drone.send_rc_control(0, 0, 0, 0)
        drone.land()
        break

cv2.destroyAllWindows()
pygame.quit()
